<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 📑 Table of Contents: E-004a Hierarchical ICD-10 (E-002 Init)

---

### 🧠 Hierarchical Two-Stage Architecture (E-004a)
Experiment objective, E-003 diagnosis, and the single architectural
fix being tested.

### 🧪 Experiment Log: Scientific Record (E-004a)
Official configuration and results.

### 🔬 Phase 1: Experiment Configuration
MLflow SQLite backend, Gold layer Parquet discovery, E-003 Stage-1
and E-002 registry path verification, Stage-2 hyperparameters.

### ⚙️ Phase 1b: Environment Setup & Imports
HuggingFace local cache, MPS fallback, seed locking, Stage-1/Stage-2
output directories.

### 📥 Phase 2: Data Loading & Label Derivation
Gold layer ingestion, billable-only filter (9,660 records), chapter
label derivation (22 classes), shared train/val/test split.

### 📊 Phase 2 Observations
Chapter distribution, skip chapter decision, training examples per
chapter.

### 🧭 Phase 3a: Stage-1 Setup
Loads Stage-1 router directly from E-003 registry — no retraining.
Tokenisation and DatasetDict construction only. Completed in seconds.

### 📊 Phase 3b: Stage-1 Evaluation
Confirms routing accuracy matches E-003 (95.4%) — identical model,
identical result.

### 🔬 Phase 4a: Stage-2 Data Preparation
Per-chapter dataset filtering, label encoders, tokenisation,
skip chapter fallback predictions. Identical to E-003.

### 📊 Phase 4b: Stage-2 Trainer Configuration
`train_chapter_resolver()` function. **Key change: initialise
from E-002 registry model instead of fresh Bio_ClinicalBERT.**
Output suppression via `contextlib` redirect and `SilentCallback`.

### 🚀 Phase 4c: Stage-2 Training Loop
19 chapter resolvers trained in priority order, each initialised
from E-002 weights. Runtime: ~2.5 hours.

### 📊 Phase 4c Cell 2: Stage-2 Val Results Summary
Per-chapter val results. Weighted avg F1 = 0.585, Acc = 69.1% —
6.6x improvement over E-003 (F1 0.089).

### 🎯 Phase 5: End-to-End Pipeline Evaluation
Full two-stage pipeline on test set. E2E accuracy 66.7%, Macro F1
0.551 — +19.8pp over E-002 flat baseline.

### 📊 E-004a Results: Interpretation
End-to-end analysis, E-002 initialisation impact, per-chapter
breakdown, comparison across all four experiments.

### 🏆 Phase 6: Registry Promotion
Stage-2 resolvers and experiment metadata saved to registry.
MLflow run closed.

---

### 🎯 Experiment Objective

Re-run the E-003 hierarchical pipeline with a single architectural
change: initialise each Stage-2 within-chapter resolver from the
**E-002 registry model** rather than fresh Bio_ClinicalBERT.

**Hypothesis confirmed.** E-002 initialisation for Stage-2 produced
a 6.3x improvement in within-chapter accuracy (69.8% vs 11.1%) and
+19.8pp end-to-end accuracy over E-002 flat (66.7% vs 46.9%).

**Official E-004a Results:**

| Stage | Metric | Value |
|---|---|---|
| Stage-1 | Chapter routing accuracy | 95.4% |
| Stage-1 | Chapter Macro F1 | 0.959 |
| Stage-2 | Within-chapter accuracy | 69.8% |
| End-to-end | Accuracy | 66.7% |
| End-to-end | Macro F1 | 0.551 |

**Full experiment comparison:**

| Experiment | Architecture | Accuracy | Macro F1 |
|---|---|---|---|
| E-001 | ICD-3 flat, 675 classes | 82.7% | 0.760 |
| E-002 | ICD-10 flat, 1,926 classes | 46.9% | 0.352 |
| E-003 | Hierarchical, fresh BERT | 10.6% | 0.070 |
| **E-004a** | **Hierarchical, E-002 init** | **66.7%** | **0.551** |

> **STATUS: COMPLETE.**
> E-004a is the best-performing ICD-10 architecture in the pipeline.
> End-to-end Accuracy = 66.7%, Macro F1 = 0.551.
> Beats E-002 flat by +19.8pp accuracy and +0.199 Macro F1.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 🧠 Hierarchical Two-Stage ICD-10 Classification (E-004a)

This notebook implements E-004a — a targeted fix to the E-003 hierarchical
pipeline. E-003 demonstrated that a dedicated Stage-1 chapter router
substantially outperforms flat chapter-level accuracy (+12.4pp), but
Stage-2 within-chapter resolvers severely underperformed due to a single
architectural flaw: initialising from fresh Bio_ClinicalBERT rather than
the E-002 registry model.

## 🎯 The Single Change

| Component | E-003 | E-004a |
|---|---|---|
| Stage-1 router | E-001 init, 10 epochs | **Reused from E-003 registry** |
| Stage-2 resolvers | Fresh Bio_ClinicalBERT | **E-002 registry model** |
| Everything else | — | Identical to E-003 |

This is a controlled single-variable experiment. Every other aspect of
the pipeline — data, splits, tokenisation, epochs, hyperparameters,
evaluation — is identical to E-003.

## 🔬 Why E-002 Initialisation Should Fix Stage-2

E-003's Stage-2 resolvers trained from scratch on chapter-filtered
subsets — Z-chapter got 1,056 records, M-chapter 888, A-chapter only 48.
Each resolver had to learn ICD-10 representations from a fraction of the
data available to E-002's flat classifier.

E-002 trained on all 7,728 records for 20 epochs and built rich
cross-chapter ICD-10 representations. Its classification head already
maps clinical note embeddings to 1,926 ICD-10 codes with 46.9% accuracy.
Fine-tuning these weights on chapter-filtered data requires only that the
model sharpen its existing within-chapter discrimination — not learn
ICD-10 from scratch.

This is the same principle that made Stage-1 successful: starting from
a model that already understands the task (E-001 for chapter routing,
E-002 for ICD-10 code resolution) and fine-tuning for the specific
sub-task.

## 🏗️ Architecture

| Stage | Task | Classes | Init | Data |
|---|---|---|---|---|
| **Stage-1** | Chapter router | 22 | E-003 registry | Not retrained |
| **Stage-2** | Within-chapter resolver | Varies | **E-002 registry** | Chapter-filtered |

## 🛠️ Technical Stack

- **Stage-1:** Loaded directly from E-003 registry — no retraining
- **Stage-2:** E-002 registry model fine-tuned per chapter
- **Framework:** HuggingFace Transformers + Trainer API
- **Metrics:** End-to-end Macro F1, Accuracy, Top-5 Accuracy
- **Tracking:** MLflow SQLite backend
- **Data:** MedSynth Gold layer Parquet — 9,660 billable records

---

**Next:** Phase 1 — Experiment configuration and registry path discovery.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 1: Experiment Configuration (E-004a)

Identical structure to notebook 04 with two key differences:

**Stage-1 is loaded from the E-003 registry** — the chapter router
achieved 95.3% test accuracy in E-003 and does not need retraining.
Reusing it ensures a clean controlled comparison with E-003 —
any difference in end-to-end accuracy is attributable solely to
the Stage-2 initialisation change.

**Stage-2 is initialised from the E-002 registry model** — the
1,926-way ICD-10 classifier trained for 20 epochs on all 7,728
billable records. Each chapter resolver fine-tunes these weights
on chapter-filtered data rather than training from scratch.

Both registry paths are verified at startup — the cell raises
`FileNotFoundError` immediately if either model is missing.

</div>

In [ ]:
# ==============================================================================
# PHASE 1: EXPERIMENT CONFIGURATION (E-004a)
# ==============================================================================
import sys
import torch
import mlflow
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# 1. BOOTSTRAP: DISCOVER PROJECT ROOT
# ------------------------------------------------------------------------------
print("🔍 Discovering project root...")

try:
    current = Path.cwd()
    while current != current.parent:
        if (current / "artifacts.yaml").exists():
            PROJECT_ROOT = current.resolve()
            break
        current = current.parent
    else:
        raise FileNotFoundError(
            "Could not find artifacts.yaml in current or parent directories."
        )
    print(f"   📍 Project root: .../{PROJECT_ROOT.name}")
except FileNotFoundError as e:
    print(f"❌ CRITICAL: {e}")
    raise

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"   📦 Project root added to sys.path")

# ------------------------------------------------------------------------------
# 2. IMPORT CONFIG
# ------------------------------------------------------------------------------
from src.config import config

if Path(config.project_root) != PROJECT_ROOT:
    raise RuntimeError(
        f"Project root mismatch!\n"
        f"  Bootstrap found: {PROJECT_ROOT}\n"
        f"  Config reports:  {config.project_root}"
    )

print(f"   ✅ Config loaded from: {config.config_path}")

# ------------------------------------------------------------------------------
# 3. MLFLOW SQLITE BACKEND
# ------------------------------------------------------------------------------
DB_PATH = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# ------------------------------------------------------------------------------
# 4. EXPERIMENT PARAMETERS
# ------------------------------------------------------------------------------
cfg = {
    "experiment_id":        "E-004a",
    "experiment_name":      "E-004a_Hierarchical_ICD10_E002Init",
    "description":          (
        "Two-stage hierarchical ICD-10 | Stage-1 from E-003 registry | "
        "Stage-2 initialised from E-002 registry model"
    ),
    "model_name":           "emilyalsentzer/Bio_ClinicalBERT",
    "payload_type":         "note_only",
    "label_scheme":         "hierarchical",
    "code_status_filter":   "billable",

    # Stage-1 — loaded from E-003 registry, not retrained
    "stage1_source":        "E-003_Hierarchical_ICD10",

    # Stage-2 — initialised from E-002 registry model
    "stage2_init_model":    "E-002_FullICD10_ClinicalBERT",
    "stage2_num_epochs":    20,
    "stage2_learning_rate": 2e-5,
    "stage2_batch_size":    16,

    "max_length":           512,
    "weight_decay":         0.01,
    "warmup_ratio":         0.1,
    "use_special_tokens":   False,
    "seed":                 42,

    # Skip chapters — same as E-003
    "skip_chapters":        ["U", "P", "Q"],

    # Priority order — same as E-003
    "stage2_priority_chapters": [
        "Z", "R", "T", "S", "B", "K", "M", "I", "L",
        "D", "E", "G", "O", "N", "J", "F", "H", "C", "A"
    ],
}

# ------------------------------------------------------------------------------
# 5. LOCATE GOLD LAYER PARQUET
# ------------------------------------------------------------------------------
gold_dir      = config.resolve_path("data", "gold")
parquet_files = sorted(gold_dir.glob("medsynth_gold_apso_*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Gold layer Parquet found in {gold_dir}\n"
        f"Please run the EDA notebook (Phase 4) first."
    )

GOLD_PARQUET_PATH = parquet_files[-1]
print(f"\n   📁 Gold layer: {GOLD_PARQUET_PATH.name}")

# ------------------------------------------------------------------------------
# 6. LOCATE REGISTRY PATHS
# ------------------------------------------------------------------------------
REGISTRY_BASE = config.resolve_path("outputs", "evaluations") / "registry"

# E-003 Stage-1 router
E003_STAGE1_PATH = (
    config.resolve_path("outputs", "evaluations")
    / cfg["stage1_source"]
    / "stage1"
)

if not E003_STAGE1_PATH.exists():
    raise FileNotFoundError(
        f"E-003 Stage-1 model not found at {E003_STAGE1_PATH}\n"
        f"Please run notebook 04 through Phase 6 first."
    )

print(f"   📁 E-003 Stage-1:  .../{E003_STAGE1_PATH.parent.name}/stage1/")

# E-002 full ICD-10 model (Stage-2 initialisation)
E002_MODEL_PATH = REGISTRY_BASE / cfg["stage2_init_model"] / "model"

if not E002_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"E-002 registry model not found at {E002_MODEL_PATH}\n"
        f"Please run notebook 03 through Phase 10 first."
    )

print(f"   📁 E-002 model:    .../{E002_MODEL_PATH.parent.name}/model/")

# ------------------------------------------------------------------------------
# 7. MLFLOW EXPERIMENT SETUP
# ------------------------------------------------------------------------------
mlflow.set_experiment(cfg["experiment_name"])

if mlflow.active_run():
    print(f"🔄 Closing active run: {mlflow.active_run().info.run_id}")
    mlflow.end_run()

run = mlflow.start_run(run_name=f"{cfg['experiment_id']}_Hierarchical_E002Init")
mlflow.log_params({k: v for k, v in cfg.items()
                   if not isinstance(v, list)})
mlflow.log_param("gold_parquet",     GOLD_PARQUET_PATH.name)
mlflow.log_param("stage1_source",    cfg["stage1_source"])
mlflow.log_param("stage2_init",      cfg["stage2_init_model"])

# ------------------------------------------------------------------------------
# 8. DEVICE SETUP
# ------------------------------------------------------------------------------
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
mlflow.log_param("device", str(device))

# ------------------------------------------------------------------------------
# 9. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 1: Experiment Configuration",
    action="e004a_config_initialised",
    details={
        "experiment_id":     cfg["experiment_id"],
        "stage1_source":     cfg["stage1_source"],
        "stage2_init_model": cfg["stage2_init_model"],
        "gold_parquet":      GOLD_PARQUET_PATH.name,
        "device":            str(device),
        "mlflow_db":         str(DB_PATH),
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n🔒 Experiment locked:  {cfg['experiment_name']}")
print(f"🧭 Stage-1 source:     E-003 registry (no retraining)")
print(f"🔬 Stage-2 init:       E-002 registry model")
print(f"🚀 Acceleration:       {device.type.upper()}")
print(f"📊 MLflow backend:     {DB_PATH.name}")
print(f"✅ Configuration ready for ingestion.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## ⚙️ Phase 1b: Environment Setup & Imports (E-004a)

Identical to notebook 04 with one addition — `E002_MODEL_PATH` is
verified in the integrity checks to confirm the E-002 registry model
is available before any training begins.

</div>

In [ ]:
# ==============================================================================
# PHASE 1b: ENVIRONMENT SETUP & IMPORTS (E-004a)
# ==============================================================================

import os
import numpy as np
from pathlib import Path
from transformers import set_seed

# Integrity checks
assert "cfg" in globals(), \
    "❌ cfg not found. Run Phase 1 first."
assert "PROJECT_ROOT" in globals(), \
    "❌ PROJECT_ROOT not found. Run Phase 1 first."
assert "GOLD_PARQUET_PATH" in globals(), \
    "❌ GOLD_PARQUET_PATH not found. Run Phase 1 first."
assert "E003_STAGE1_PATH" in globals(), \
    "❌ E003_STAGE1_PATH not found. Run Phase 1 first."
assert "E002_MODEL_PATH" in globals(), \
    "❌ E002_MODEL_PATH not found. Run Phase 1 first."

# HuggingFace local cache
HF_CACHE_DIR = PROJECT_ROOT / "data" / "cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]                = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"]           = str(HF_CACHE_DIR)
os.environ["HF_HUB_READ_TIMEOUT"]    = "120"
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Imports
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

# Seed
active_seed = cfg.get("seed", 42)
set_seed(active_seed)

# Experiment output directories
EXP_DIR      = config.resolve_path("outputs", "evaluations") / cfg["experiment_name"]
STAGE1_DIR   = EXP_DIR / "stage1"
STAGE2_DIR   = EXP_DIR / "stage2"
REGISTRY_DIR = config.resolve_path("outputs", "evaluations") / "registry" / cfg["experiment_name"]

for d in [EXP_DIR, STAGE1_DIR, STAGE2_DIR, REGISTRY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["TENSORBOARD_LOGGING_DIR"] = str(EXP_DIR / "tensorboard")

print(f"✅ Phase 1b complete")
print(f"   HF cache:        {HF_CACHE_DIR}")
print(f"   Experiment dir:  {EXP_DIR}")
print(f"   Stage-1 dir:     {STAGE1_DIR}")
print(f"   Stage-2 dir:     {STAGE2_DIR}")
print(f"   Seed:            {active_seed}")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📥 Phase 2: Data Loading & Label Derivation (E-004a)

Identical to notebook 04. Same Gold layer, same billable filter, same
chapter label derivation, same train/val/test split. The split uses the
same random seed as E-003 ensuring identical train/val/test record
assignment — a controlled comparison requirement.

</div>

In [ ]:
# ==============================================================================
# PHASE 2: DATA LOADING & LABEL DERIVATION (E-004a)
# ==============================================================================

import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split

print(f"📂 Loading Gold layer: {GOLD_PARQUET_PATH.name}")

df_gold = pl.read_parquet(GOLD_PARQUET_PATH)

print(f"   ✅ Loaded: {len(df_gold):,} records, {len(df_gold.columns)} columns")

required_cols = {'id', 'apso_note', 'standard_icd10', 'code_status'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(f"❌ Gold layer missing required columns: {missing_cols}")
print(f"   ✅ All required columns present")

# Filter to billable
print(f"\n📊 Full dataset code_status breakdown:")
for row in (df_gold.group_by("code_status")
            .agg(pl.len().alias("count"))
            .sort("count", descending=True)
            .iter_rows(named=True)):
    pct = row['count'] / len(df_gold) * 100
    print(f"   {row['code_status']:20s}: {row['count']:,} ({pct:.1f}%)")

df_gold = df_gold.filter(pl.col("code_status") == cfg["code_status_filter"])
print(f"\n   ✅ Filtered to '{cfg['code_status_filter']}': {len(df_gold):,} records")

# Derive chapter labels (Stage-1)
print(f"\n🔧 Deriving chapter labels (Stage-1)...")
df_gold = df_gold.with_columns(
    pl.col("standard_icd10").str.slice(0, 1).alias("chapter_label")
)

chapters     = sorted(df_gold["chapter_label"].unique().to_list())
chapter2id   = {ch: i for i, ch in enumerate(chapters)}
id2chapter   = {i: ch for ch, i in chapter2id.items()}
num_chapters = len(chapters)

df_gold = df_gold.with_columns(
    pl.col("chapter_label")
    .replace(list(chapter2id.keys()), [chapter2id[k] for k in chapter2id.keys()])
    .cast(pl.Int64)
    .alias("chapter_id")
)

print(f"   ✅ Chapter labels derived: {num_chapters} ICD-10 chapters")

# Encode full ICD-10 labels (Stage-2)
print(f"\n🔧 Encoding ICD-10 labels (Stage-2)...")
all_icd10_labels = sorted(df_gold["standard_icd10"].unique().to_list())
icd10_label2id   = {label: idx for idx, label in enumerate(all_icd10_labels)}
icd10_id2label   = {idx: label for label, idx in icd10_label2id.items()}
num_icd10_labels = len(icd10_label2id)

df_gold = df_gold.with_columns(
    pl.col("standard_icd10")
    .replace(list(icd10_label2id.keys()),
             [icd10_label2id[k] for k in icd10_label2id.keys()])
    .cast(pl.Int64)
    .alias("icd10_id")
)
print(f"   ✅ ICD-10 labels encoded: {num_icd10_labels:,} classes")

# Train/val/test split (80/10/10) — stratified by chapter
print(f"\n✂️  Splitting dataset (80/10/10) — stratified by chapter...")
df_pd = df_gold.select([
    "id", "apso_note",
    "standard_icd10", "icd10_id",
    "chapter_label", "chapter_id",
    "code_status"
]).to_pandas()

train_df, temp_df = train_test_split(
    df_pd,
    test_size=0.2,
    stratify=df_pd["chapter_id"],
    random_state=cfg["seed"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=None,
    random_state=cfg["seed"]
)

print(f"   ✅ Split complete:")
print(f"      Train: {len(train_df):,} ({len(train_df)/len(df_pd)*100:.1f}%) — stratified by chapter")
print(f"      Val:   {len(val_df):,}  ({len(val_df)/len(df_pd)*100:.1f}%) — random")
print(f"      Test:  {len(test_df):,}  ({len(test_df)/len(df_pd)*100:.1f}%) — random")

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"      {name:5s} chapter coverage: {df['chapter_label'].nunique()}/{num_chapters}")

mlflow.log_params({
    "num_chapters":        num_chapters,
    "num_icd10_labels":    num_icd10_labels,
    "train_size":          len(train_df),
    "val_size":            len(val_df),
    "test_size":           len(test_df),
})

config.log_event(
    phase="Phase 2: Data Loading & Label Derivation",
    action="data_loaded_and_split",
    details={
        "gold_parquet":     GOLD_PARQUET_PATH.name,
        "total_records":    len(df_gold),
        "num_chapters":     num_chapters,
        "num_icd10_labels": num_icd10_labels,
        "train_size":       len(train_df),
        "val_size":         len(val_df),
        "test_size":        len(test_df),
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 2 complete: {num_chapters}-way Stage-1 + {num_icd10_labels:,}-way Stage-2 ready")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧭 Phase 3a: Stage-1 Setup — Load from E-003 Registry

Loads the Stage-1 chapter router trained in notebook 04 directly from
the E-003 registry. No retraining required — the model achieved 95.3%
test routing accuracy in E-003 and is reused unchanged.

Reusing the identical Stage-1 model ensures the E-003 vs E-004a
comparison is controlled — any difference in end-to-end accuracy is
attributable solely to the Stage-2 initialisation change.

A dummy `Trainer` is constructed from the loaded model to enable
`trainer.predict()` calls in Phase 3b evaluation and Phase 5
end-to-end inference.

</div>

In [ ]:
# ==============================================================================
# PHASE 3a: STAGE-1 SETUP — LOAD FROM E-003 REGISTRY
# ==============================================================================
# Stage-1 was trained in notebook 04 and achieved 95.3% test routing accuracy.
# No retraining needed — loading directly saves ~70 minutes.
# ==============================================================================

import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from src.evaluation import hf_compute_metrics

# ------------------------------------------------------------------------------
# 1. LOAD TOKENIZER FROM E-003 STAGE-1
# ------------------------------------------------------------------------------
print(f"📥 Loading tokenizer from E-003 Stage-1...")

tokenizer = AutoTokenizer.from_pretrained(
    str(E003_STAGE1_PATH / "model"),
    cache_dir=str(HF_CACHE_DIR)
)
print(f"   ✅ Tokenizer loaded: vocab size {len(tokenizer):,}")

# ------------------------------------------------------------------------------
# 2. BUILD STAGE-1 DATASETDICT (chapter-level labels)
# ------------------------------------------------------------------------------
print(f"\n🔧 Building Stage-1 DatasetDict (22-way chapter classification)...")

class_names_chapter = sorted(list(chapter2id.keys()))
features_stage1 = Features({
    'text':  Value('string'),
    'label': ClassLabel(names=class_names_chapter)
})

stage1_datasets = DatasetDict({
    "train": Dataset.from_dict(
        {"text": train_df["apso_note"].tolist(),
         "label": train_df["chapter_id"].tolist()},
        features=features_stage1
    ),
    "val": Dataset.from_dict(
        {"text": val_df["apso_note"].tolist(),
         "label": val_df["chapter_id"].tolist()},
        features=features_stage1
    ),
    "test": Dataset.from_dict(
        {"text": test_df["apso_note"].tolist(),
         "label": test_df["chapter_id"].tolist()},
        features=features_stage1
    ),
})

print(f"   ✅ Stage-1 DatasetDict:")
for split in ["train", "val", "test"]:
    print(f"      {split:5s}: {len(stage1_datasets[split]):,} records, "
          f"{stage1_datasets[split].features['label'].num_classes} classes")

# ------------------------------------------------------------------------------
# 3. TOKENISE STAGE-1 DATASETS
# ------------------------------------------------------------------------------
print(f"\n🔄 Tokenising Stage-1 dataset (max_length={cfg['max_length']})...")

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg["max_length"]
    )

stage1_tokenized = stage1_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["text"]
)
stage1_tokenized.set_format("torch")
print(f"   ✅ Tokenisation complete")

# Token 0 verification
import torch
sample_ids    = stage1_tokenized["train"][0]["input_ids"]
first_content = tokenizer.decode(sample_ids[1:15], skip_special_tokens=True)
print(f"   🔬 Content tokens 1–14: {first_content}")

# ------------------------------------------------------------------------------
# 4. LOAD STAGE-1 MODEL FROM E-003 REGISTRY
# ------------------------------------------------------------------------------
print(f"\n📥 Loading Stage-1 model from E-003 registry...")

STAGE1_MODEL_DIR = E003_STAGE1_PATH / "model"

stage1_model = AutoModelForSequenceClassification.from_pretrained(
    str(STAGE1_MODEL_DIR)
)
stage1_model.to(device)
stage1_model.eval()

print(f"   ✅ Stage-1 model loaded: {stage1_model.num_labels}-way classifier")
print(f"   ✅ Device: {next(stage1_model.parameters()).device}")

# Load chapter mapping
with open(E003_STAGE1_PATH / "chapter_mapping.json") as f:
    ch_map = json.load(f)
print(f"   ✅ Chapter mapping loaded: {ch_map['num_chapters']} chapters")

# ------------------------------------------------------------------------------
# 5. BUILD DUMMY TRAINER FOR PREDICT()
# ------------------------------------------------------------------------------
_dummy_args = TrainingArguments(
    output_dir            = str(STAGE1_DIR / "checkpoints"),
    fp16                  = False,
    dataloader_pin_memory = False,
    log_level             = "error",
    disable_tqdm          = True,
    report_to             = "none",
)
stage1_trainer = Trainer(
    model            = stage1_model,
    args             = _dummy_args,
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
)

print(f"\n✅ Phase 3a complete — Stage-1 loaded from E-003 registry (no retraining)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 3b: Stage-1 Evaluation

Verifies that the loaded Stage-1 model produces routing accuracy
consistent with E-003 (95.3% test accuracy). This confirms the model
loaded correctly before Stage-2 training begins.

Expected: 95.3–95.4% test routing accuracy — identical to E-003 since
the same model weights are used.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b: STAGE-1 TEST SET EVALUATION
# ==============================================================================
import numpy as np

print(f"🔍 Evaluating Stage-1 router on test set ({len(stage1_tokenized['test']):,} records)...")

stage1_test_output = stage1_trainer.predict(stage1_tokenized["test"])
stage1_y_true      = stage1_test_output.label_ids
stage1_y_pred      = np.argmax(stage1_test_output.predictions, axis=-1)

print(f"\n📊 Stage-1 Test Results:")
for k, v in stage1_test_output.metrics.items():
    print(f"   {k}: {v:.4f}")

print(f"\n📊 Per-chapter routing accuracy (test set):")
print(f"   {'Chapter':8s}  {'True':>6s}  {'Correct':>8s}  {'Accuracy':>9s}")
print(f"   {'─'*36}")

chapter_test_stats = {}
for ch_id, ch_name in sorted(id2chapter.items()):
    mask      = stage1_y_true == ch_id
    n_true    = mask.sum()
    n_correct = (stage1_y_pred[mask] == ch_id).sum()
    accuracy  = n_correct / n_true if n_true > 0 else 0
    chapter_test_stats[ch_name] = {
        "n_true":    int(n_true),
        "n_correct": int(n_correct),
        "accuracy":  float(accuracy)
    }
    print(f"   {ch_name:8s}  {n_true:>6,}  {n_correct:>8,}  {accuracy:>8.1%}")

stage1_test_accuracy = (stage1_y_pred == stage1_y_true).mean()
stage1_test_f1       = stage1_test_output.metrics.get("test_macro_f1", 0)

print(f"\n📊 Routing error budget for Stage-2:")
print(f"   Correctly routed:   {stage1_test_accuracy:.1%}")
print(f"   Misrouted:          {1-stage1_test_accuracy:.1%}")

mlflow.log_metrics({
    "stage1_test_accuracy": float(stage1_test_accuracy),
    "stage1_test_macro_f1": float(stage1_test_f1),
    "stage1_test_top5":     stage1_test_output.metrics.get(
                            "test_top_5_accuracy", 0),
})

import json
with open(STAGE1_DIR / "chapter_test_stats.json", "w") as f:
    json.dump(chapter_test_stats, f, indent=4)

config.log_event(
    phase="Phase 3b: Stage-1 Test Evaluation",
    action="stage1_test_evaluation_complete",
    details={
        "test_accuracy": float(stage1_test_accuracy),
        "test_macro_f1": float(stage1_test_f1),
        "source":        "E-003 registry (no retraining)",
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 3b complete — Stage-1 routing verified")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 4a: Stage-2 Data Preparation (E-004a)

Identical to notebook 04. Filters train/val/test splits by chapter,
builds per-chapter label encoders, and tokenises all 19 trainable
chapter datasets.

Skip chapters (U, P, Q) receive fallback predictions — unchanged from
E-003.

</div>

In [ ]:
# ==============================================================================
# PHASE 4a: STAGE-2 DATA PREPARATION (E-004a)
# ==============================================================================

import json
import numpy as np
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

SKIP_CHAPTERS = {"U", "P", "Q"}

skip_chapter_defaults = {}
for ch in SKIP_CHAPTERS:
    ch_train = train_df[train_df["chapter_label"] == ch]
    if len(ch_train) > 0:
        most_frequent = ch_train["standard_icd10"].value_counts().index[0]
        skip_chapter_defaults[ch] = most_frequent
    else:
        skip_chapter_defaults[ch] = None

print(f"⏭️  Skip chapters (fallback predictions):")
for ch, code in skip_chapter_defaults.items():
    print(f"   {ch}: → {code}")

TRAIN_CHAPTERS = [ch for ch in chapters if ch not in SKIP_CHAPTERS]
print(f"\n✅ Trainable chapters: {len(TRAIN_CHAPTERS)}")
print(f"   {TRAIN_CHAPTERS}")

print(f"\n🔧 Preparing per-chapter datasets...")

chapter_datasets   = {}
chapter_label_maps = {}

for ch in TRAIN_CHAPTERS:
    ch_train = train_df[train_df["chapter_label"] == ch].copy()
    ch_val   = val_df[val_df["chapter_label"] == ch].copy()
    ch_test  = test_df[test_df["chapter_label"] == ch].copy()

    if len(ch_train) == 0:
        print(f"   ⚠️  {ch}: no training records — skipping")
        continue

    ch_labels    = sorted(ch_train["standard_icd10"].unique().tolist())
    ch_label2id  = {label: i for i, label in enumerate(ch_labels)}
    ch_id2label  = {i: label for label, i in ch_label2id.items()}
    ch_num_labels = len(ch_label2id)

    ch_train["ch_label_id"] = ch_train["standard_icd10"].map(ch_label2id)
    ch_val["ch_label_id"]   = ch_val["standard_icd10"].map(ch_label2id)
    ch_test["ch_label_id"]  = ch_test["standard_icd10"].map(ch_label2id)

    ch_val  = ch_val.dropna(subset=["ch_label_id"])
    ch_test = ch_test.dropna(subset=["ch_label_id"])

    ch_train["ch_label_id"] = ch_train["ch_label_id"].astype(int)
    ch_val["ch_label_id"]   = ch_val["ch_label_id"].astype(int)
    ch_test["ch_label_id"]  = ch_test["ch_label_id"].astype(int)

    ch_features = Features({
        'text':  Value('string'),
        'label': ClassLabel(names=ch_labels)
    })

    datasets_dict = {"train": Dataset.from_dict(
        {"text": ch_train["apso_note"].tolist(),
         "label": ch_train["ch_label_id"].tolist()},
        features=ch_features
    )}

    if len(ch_val) > 0:
        datasets_dict["val"] = Dataset.from_dict(
            {"text": ch_val["apso_note"].tolist(),
             "label": ch_val["ch_label_id"].tolist()},
            features=ch_features
        )

    if len(ch_test) > 0:
        datasets_dict["test"] = Dataset.from_dict(
            {"text": ch_test["apso_note"].tolist(),
             "label": ch_test["ch_label_id"].tolist()},
            features=ch_features
        )

    chapter_datasets[ch]   = DatasetDict(datasets_dict)
    chapter_label_maps[ch] = {
        "label2id":   ch_label2id,
        "id2label":   {str(k): v for k, v in ch_id2label.items()},
        "num_labels": ch_num_labels,
        "chapter":    ch,
    }

    print(f"   {ch:4s}: {len(ch_train):>4,} train | {len(ch_val):>3,} val | "
          f"{len(ch_test):>3,} test | {ch_num_labels:>4,} classes")

print(f"\n🔄 Tokenising all chapter datasets...")

def preprocess_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg["max_length"]
    )

chapter_tokenized = {}
for ch in TRAIN_CHAPTERS:
    if ch not in chapter_datasets:
        continue
    ch_tok = chapter_datasets[ch].map(
        preprocess_fn, batched=True, remove_columns=["text"]
    )
    ch_tok.set_format("torch")
    chapter_tokenized[ch] = ch_tok

print(f"   ✅ Tokenised {len(chapter_tokenized)} chapter datasets")

STAGE2_LABEL_MAP_PATH = STAGE2_DIR / "chapter_label_maps.json"
with open(STAGE2_LABEL_MAP_PATH, "w") as f:
    json.dump(chapter_label_maps, f, indent=4)
print(f"\n   ✅ Chapter label maps saved: {STAGE2_LABEL_MAP_PATH.name}")

config.log_event(
    phase="Phase 4a: Stage-2 Data Preparation",
    action="chapter_datasets_prepared",
    details={
        "trainable_chapters": TRAIN_CHAPTERS,
        "skip_chapters":      list(SKIP_CHAPTERS),
        "skip_defaults":      skip_chapter_defaults,
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 4a complete: {len(chapter_tokenized)} chapter datasets ready")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4b: Stage-2 Trainer Configuration (E-004a)

Identical to notebook 04 with one critical change: each chapter resolver
is initialised from the **E-002 registry model** rather than fresh
Bio_ClinicalBERT.

### Why E-002 Initialisation Should Fix Stage-2

E-003's resolvers trained from scratch on chapter-filtered subsets and
had to learn ICD-10 representations from a fraction of the data available
to E-002's flat classifier. E-002 trained on all 7,728 records for 20
epochs and already maps clinical note embeddings to 1,926 ICD-10 codes
with 46.9% accuracy.

Fine-tuning E-002 weights on chapter-filtered data requires only that
the model sharpen its existing within-chapter discrimination — not learn
ICD-10 from scratch. The classification head is replaced with a
chapter-specific head sized to the chapter's ICD-10 classes.

### What Changes vs E-003
```python
# E-003 (fresh Bio_ClinicalBERT):
AutoModelForSequenceClassification.from_pretrained(cfg["model_name"], ...)

# E-004a (E-002 registry model):
AutoModelForSequenceClassification.from_pretrained(str(E002_MODEL_PATH), ...)
```

Everything else — epochs, learning rate, batch size, metrics,
checkpointing — is identical to E-003.

</div>

In [ ]:
# ==============================================================================
# PHASE 4b: STAGE-2 TRAINER CONFIGURATION (E-004a)
# ==============================================================================

import os
import io
import contextlib
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback
)
from src.evaluation import hf_compute_metrics

STAGE2_TB_DIR = STAGE2_DIR / "tensorboard"
STAGE2_TB_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TENSORBOARD_LOGGING_DIR"] = str(STAGE2_TB_DIR)

class SilentCallback(TrainerCallback):
    """Suppresses all Trainer output during training."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        pass
    def on_epoch_end(self, args, state, control, **kwargs):
        pass
    def on_evaluate(self, args, state, control, **kwargs):
        pass

def train_chapter_resolver(chapter: str) -> dict:
    """
    Trains a within-chapter ICD-10 resolver for a single chapter.
    KEY CHANGE vs E-003: initialised from E-002 registry model,
    not fresh Bio_ClinicalBERT.
    Prints one summary line per chapter only.
    """
    if chapter not in chapter_tokenized:
        print(f"   ⚠️  {chapter}: no tokenised data — skipping")
        return {}

    ch_tok        = chapter_tokenized[chapter]
    ch_label_map  = chapter_label_maps[chapter]
    ch_num_labels = ch_label_map["num_labels"]
    ch_label2id   = ch_label_map["label2id"]
    ch_id2label   = {int(k): v for k, v in ch_label_map["id2label"].items()}

    # ------------------------------------------------------------------
    # 1. LOAD FROM E-002 REGISTRY — suppress load report output
    # ------------------------------------------------------------------
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_model = AutoModelForSequenceClassification.from_pretrained(
            str(E002_MODEL_PATH),          # ← KEY CHANGE: E-002 not fresh BERT
            num_labels=ch_num_labels,
            id2label=ch_id2label,
            label2id=ch_label2id,
            ignore_mismatched_sizes=True,  # replaces 1,926-way head with chapter head
            cache_dir=str(HF_CACHE_DIR)
        )
    ch_model.to(device)

    assert ch_model.num_labels == ch_num_labels, \
        f"❌ Head mismatch: {ch_model.num_labels} vs {ch_num_labels}"

    # ------------------------------------------------------------------
    # 2. TRAINING ARGUMENTS
    # ------------------------------------------------------------------
    ch_checkpoint_dir = STAGE2_DIR / chapter / "checkpoints"
    ch_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    ch_total_steps  = (
        len(ch_tok["train"]) // cfg["stage2_batch_size"]
    ) * cfg["stage2_num_epochs"]
    ch_warmup_steps = max(1, int(cfg["warmup_ratio"] * ch_total_steps))

    ch_args = TrainingArguments(
        output_dir              = str(ch_checkpoint_dir),
        eval_strategy           = "epoch" if "val" in ch_tok else "no",
        save_strategy           = "epoch",
        load_best_model_at_end  = "val" in ch_tok,
        metric_for_best_model   = "macro_f1",
        greater_is_better       = True,
        save_total_limit        = 2,
        num_train_epochs            = cfg["stage2_num_epochs"],
        per_device_train_batch_size = cfg["stage2_batch_size"],
        learning_rate               = cfg["stage2_learning_rate"],
        weight_decay                = cfg["weight_decay"],
        warmup_steps                = ch_warmup_steps,
        logging_steps               = ch_total_steps + 1,
        report_to                   = ["tensorboard"],
        seed                        = cfg["seed"],
        fp16                        = False,
        dataloader_pin_memory       = False,
        log_level                   = "error",
        disable_tqdm                = True,
    )

    # ------------------------------------------------------------------
    # 3. TRAINER — with SilentCallback
    # ------------------------------------------------------------------
    ch_trainer = Trainer(
        model            = ch_model,
        args             = ch_args,
        train_dataset    = ch_tok["train"],
        eval_dataset     = ch_tok.get("val", None),
        processing_class = tokenizer,
        data_collator    = DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics  = hf_compute_metrics if "val" in ch_tok else None,
        callbacks        = [SilentCallback()],
    )

    # ------------------------------------------------------------------
    # 4. TRAIN — suppress all stdout/stderr
    # ------------------------------------------------------------------
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_result = ch_trainer.train()

    # ------------------------------------------------------------------
    # 5. EXTRACT BEST VAL METRICS
    # ------------------------------------------------------------------
    ch_val_metrics = {}
    if "val" in ch_tok:
        for log in ch_trainer.state.log_history:
            if "eval_macro_f1" in log:
                ch_val_metrics = log

    # ------------------------------------------------------------------
    # 6. SAVE MODEL AND LABEL MAP — suppress shard output
    # ------------------------------------------------------------------
    ch_model_dir = STAGE2_DIR / chapter / "model"
    ch_model_dir.mkdir(parents=True, exist_ok=True)

    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_trainer.save_model(str(ch_model_dir))
        tokenizer.save_pretrained(str(ch_model_dir))

    import json
    with open(ch_model_dir / "label_map.json", "w") as f:
        json.dump(ch_label_map, f, indent=4)

    # ------------------------------------------------------------------
    # 7. ONE SUMMARY LINE PER CHAPTER
    # ------------------------------------------------------------------
    val_f1   = ch_val_metrics.get("eval_macro_f1", 0)
    val_acc  = ch_val_metrics.get("eval_accuracy", 0)
    val_top5 = ch_val_metrics.get("eval_top_5_accuracy", 0)

    print(f"   ✅ {chapter:4s} | {ch_num_labels:4d} classes | "
          f"loss {ch_result.training_loss:.3f} | "
          f"F1 {val_f1:.3f} | acc {val_acc:.3f} | top5 {val_top5:.3f}")

    return {
        "chapter":      chapter,
        "num_labels":   ch_num_labels,
        "train_loss":   ch_result.training_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc,
        "val_top5":     val_top5,
        "model_path":   str(ch_model_dir),
    }

# ------------------------------------------------------------------
# MONITORING
# ------------------------------------------------------------------
print(f"✅ Stage-2 trainer function defined")
print(f"   19 chapter resolvers will be trained")
print(f"   Init:    E-002 registry model (KEY CHANGE vs E-003)")
print(f"   Epochs:  {cfg['stage2_num_epochs']}, "
      f"lr={cfg['stage2_learning_rate']}, batch={cfg['stage2_batch_size']}")
print(f"   Output:  one summary line per chapter")
print(f"   Suppression: contextlib redirect + SilentCallback")

print(f"\n{'='*70}")
print(f"📈 TENSORBOARD (Stage-2 — all chapters):")
print(f"   tensorboard --logdir '{STAGE2_TB_DIR}' --port 6006")
print(f"\n📊 MLFLOW UI:")
print(f"   mlflow ui --backend-store-uri sqlite:///{PROJECT_ROOT}/mlflow.db --port 5001")
print(f"{'='*70}")
print(f"\n✅ Phase 4b complete — run test cell then Phase 4c Cell 1")

In [ ]:
# ==============================================================================
# PHASE 4b TEST: VERIFY OUTPUT SUPPRESSION (single chapter)
# ==============================================================================
print("🧪 Testing output suppression with Chapter A (smallest chapter)...")
print("   If working correctly you will see exactly ONE summary line below:")
print()
test_result = train_chapter_resolver("A")
print()
print(f"✅ Test complete — output suppression {'WORKING ✅' if test_result else 'FAILED ❌'}")

In [ ]:
# ==============================================================================
# PHASE 4c CELL 1: STAGE-2 TRAINING LOOP (E-004a)
# ==============================================================================
import json
from datetime import datetime

CHAPTER_PRIORITY = [
    "Z", "R", "T", "S", "B", "K", "M", "I", "L",
    "D", "E", "G", "O", "N", "J", "F", "H", "C", "A",
]

assert set(CHAPTER_PRIORITY) == set(TRAIN_CHAPTERS), \
    f"❌ Chapter mismatch: {set(CHAPTER_PRIORITY) ^ set(TRAIN_CHAPTERS)}"

print(f"🚀 Stage-2 Training Loop — {len(CHAPTER_PRIORITY)} chapter resolvers")
print(f"   Init:     E-002 registry model (KEY CHANGE vs E-003)")
print(f"   Started:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n   {'Ch':4s}  {'Classes':>7s}  {'Loss':>6s}  {'F1':>6s}  "
      f"{'Acc':>6s}  {'Top5':>6s}")
print(f"   {'─'*50}")

stage2_results  = {}
stage2_failures = []

for i, chapter in enumerate(CHAPTER_PRIORITY):
    try:
        result = train_chapter_resolver(chapter)
        if result:
            stage2_results[chapter] = result
    except Exception as e:
        print(f"   ❌ {chapter}: FAILED — {e}")
        stage2_failures.append({"chapter": chapter, "error": str(e)})
        continue

print(f"\n   Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Trained:  {len(stage2_results)}/{len(CHAPTER_PRIORITY)}")
print(f"   Failed:   {len(stage2_failures)}")

STAGE2_RESULTS_PATH = STAGE2_DIR / "stage2_results.json"
with open(STAGE2_RESULTS_PATH, "w") as f:
    json.dump({
        "results":       stage2_results,
        "failures":      stage2_failures,
        "skip_chapters": skip_chapter_defaults,
    }, f, indent=4)

print(f"\n   ✅ Results saved to: {STAGE2_RESULTS_PATH.name}")
print(f"✅ Cell 1 complete — run Cell 2 for summary table")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 4c Cell 2: Stage-2 Val Results Summary (E-004a)

### The E-002 Initialisation Works

| Metric | E-003 (fresh BERT) | E-004a (E-002 init) | Improvement |
|---|---|---|---|
| Weighted val F1 | 0.089 | 0.585 | +0.496 (+557%) |
| Weighted val Acc | 13.3% | 69.1% | +55.8pp |

The E-002 initialisation produced a **6.6x improvement in Macro F1**
and a **+55.8pp improvement in accuracy** across all 19 chapter
resolvers. This confirms the diagnosis from E-003 — the failure was
entirely attributable to training from scratch, not to the hierarchical
architecture itself.

### Per-Chapter Highlights

**Perfect or near-perfect resolvers:**
- T (F1 1.000, Acc 100%) — 15 classes, tiny val set
- A (F1 1.000, Acc 100%) — 12 classes, tiny val set
- B (F1 0.917, Acc 94.4%) — 31 classes
- R (F1 0.730, Acc 84.5%) — 196 classes — remarkable given size
- N (F1 0.753, Acc 84.6%) — 105 classes

**Strong resolvers (F1 > 0.60):**
I (0.675), D (0.662), L (0.656), G (0.656), H (0.683), S (0.649),
M (0.579), O (0.575), J (0.570), F (0.569)

**Weaker resolvers:**
- Z (F1 0.287, Acc 39.7%) — 263 classes, the hardest chapter.
  Still dramatically better than E-003's Z (F1 0.117, Acc 18.3%)
- K (F1 0.443, Acc 58.7%) — 116 classes
- E (F1 0.492, Acc 60.0%) — 76 classes

**Top-5 Accuracy is exceptional across the board** — 15 of 19
chapters achieve Top-5 > 90%, with 4 chapters at 100%. For a
clinical coding assistance tool this is the most practically
meaningful metric.

### What This Means for End-to-End Performance

With Stage-1 routing accuracy of 95.4% and Stage-2 weighted val
accuracy of 69.1%, the expected end-to-end accuracy is approximately:

> 0.954 × 0.691 ≈ **65.9%**

This would represent a **+19pp improvement over E-002's 46.9%**
and dramatically exceeds E-003's 10.6%. The E-004a hypothesis is
confirmed at the Stage-2 level — end-to-end evaluation in Phase 5
will give the definitive result.

</div>

In [ ]:
# ==============================================================================
# PHASE 4c CELL 2: SUMMARY TABLE (E-004a)
# ==============================================================================
import json

with open(STAGE2_DIR / "stage2_results.json") as f:
    saved = json.load(f)

results  = saved["results"]
failures = saved["failures"]

print(f"📊 Stage-2 Val Results Summary (E-004a — E-002 Init):")
print(f"   {'Ch':4s}  {'Classes':>7s}  {'Val F1':>8s}  {'Val Acc':>8s}  {'Top-5':>8s}")
print(f"   {'─'*48}")

weighted_f1  = 0
weighted_acc = 0
total_n      = 0

for ch in CHAPTER_PRIORITY:
    if ch not in results:
        print(f"   {ch:4s}  {'—':>7s}  {'FAILED':>8s}  {'FAILED':>8s}  {'FAILED':>8s}")
        continue
    r    = results[ch]
    f1   = r.get("val_macro_f1", 0)
    acc  = r.get("val_accuracy", 0)
    top5 = r.get("val_top5", 0)
    n    = r.get("num_labels", 0)
    weighted_f1  += f1 * n
    weighted_acc += acc * n
    total_n      += n
    print(f"   {ch:4s}  {r['num_labels']:>7,}  {f1:>8.4f}  {acc:>8.4f}  {top5:>8.4f}")

if total_n > 0:
    print(f"\n   {'WTD':4s}  {'':>7s}  "
          f"{weighted_f1/total_n:>8.4f}  {weighted_acc/total_n:>8.4f}")

if failures:
    print(f"\n⚠️  Failed chapters:")
    for f in failures:
        print(f"   {f['chapter']}: {f['error']}")

print(f"\n   Skip chapters: {saved['skip_chapters']}")
print(f"\n   E-003 weighted avg for comparison: F1 0.089 | Acc 0.133")
print(f"✅ Cell 2 complete — run Cell 3 for MLflow + audit")

In [ ]:
# ==============================================================================
# PHASE 4c CELL 3: MLFLOW LOGGING + AUDIT TRAIL (E-004a)
# ==============================================================================
import json

with open(STAGE2_DIR / "stage2_results.json") as f:
    saved = json.load(f)

results  = saved["results"]
failures = saved["failures"]

for ch, r in results.items():
    mlflow.log_metrics({
        f"stage2_{ch}_val_f1":       r.get("val_macro_f1", 0),
        f"stage2_{ch}_val_accuracy":  r.get("val_accuracy", 0),
        f"stage2_{ch}_train_loss":    r.get("train_loss", 0),
        f"stage2_{ch}_num_labels":    r.get("num_labels", 0),
    })

total_n      = sum(r.get("num_labels", 0) for r in results.values())
weighted_f1  = sum(r.get("val_macro_f1", 0) * r.get("num_labels", 0)
                   for r in results.values())
weighted_acc = sum(r.get("val_accuracy", 0) * r.get("num_labels", 0)
                   for r in results.values())

avg_f1  = weighted_f1  / total_n if total_n > 0 else 0
avg_acc = weighted_acc / total_n if total_n > 0 else 0

mlflow.log_metrics({
    "stage2_chapters_trained": len(results),
    "stage2_chapters_failed":  len(failures),
    "stage2_weighted_val_f1":  avg_f1,
    "stage2_weighted_val_acc": avg_acc,
})

config.log_event(
    phase="Phase 4c: Stage-2 Training Loop",
    action="stage2_training_complete",
    details={
        "chapters_trained":  len(results),
        "chapters_failed":   len(failures),
        "weighted_val_f1":   avg_f1,
        "weighted_val_acc":  avg_acc,
        "stage2_init_model": cfg["stage2_init_model"],
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"   Chapters trained: {len(results)}/19")
print(f"   Weighted val F1:  {avg_f1:.4f}")
print(f"   Weighted val Acc: {avg_acc:.4f}")
print(f"\n📝 Audit trail updated")
print(f"✅ Phase 4c complete — proceed to Phase 5")

In [ ]:
# ==============================================================================
# PHASE 5: END-TO-END PIPELINE EVALUATION (E-004a)
# ==============================================================================

import json
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------------------------
# 1. LOAD ALL STAGE-2 RESOLVER MODELS
# ------------------------------------------------------------------------------
print(f"📥 Loading Stage-2 resolver models...")

stage2_models     = {}
stage2_tokenizers = {}
stage2_label_maps = {}

for ch in CHAPTER_PRIORITY:
    ch_model_dir = STAGE2_DIR / ch / "model"
    if not ch_model_dir.exists():
        print(f"   ⚠️  Chapter {ch}: model not found — skipping")
        continue

    with open(ch_model_dir / "label_map.json") as f:
        lmap = json.load(f)

    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        ch_model = AutoModelForSequenceClassification.from_pretrained(
            str(ch_model_dir)
        )
    ch_model.to(device)
    ch_model.eval()

    ch_tokenizer = AutoTokenizer.from_pretrained(str(ch_model_dir))

    stage2_models[ch]     = ch_model
    stage2_tokenizers[ch] = ch_tokenizer
    stage2_label_maps[ch] = lmap

    print(f"   ✅ {ch}: {lmap['num_labels']} classes loaded")

print(f"\n   ✅ {len(stage2_models)}/19 resolvers loaded into memory")

# ------------------------------------------------------------------------------
# 2. STAGE-1 PREDICTIONS ON TEST SET
# ------------------------------------------------------------------------------
print(f"\n🧭 Running Stage-1 chapter routing on test set...")

stage1_model.eval()
stage1_test_preds = []

with torch.no_grad():
    for i in range(0, len(stage1_tokenized["test"]), 32):
        batch = {
            k: stage1_tokenized["test"][i:i+32][k].to(device)
            for k in ["input_ids", "attention_mask", "token_type_ids"]
        }
        outputs = stage1_model(**batch)
        preds   = torch.argmax(outputs.logits, dim=-1)
        stage1_test_preds.extend(preds.cpu().numpy().tolist())

stage1_test_preds = np.array(stage1_test_preds)
true_chapter_ids  = np.array(stage1_tokenized["test"]["label"])
true_icd10_codes  = test_df["standard_icd10"].tolist()

stage1_routing_acc = (stage1_test_preds == true_chapter_ids).mean()
print(f"   ✅ Stage-1 routing accuracy: {stage1_routing_acc:.1%}")

# ------------------------------------------------------------------------------
# 3. END-TO-END PIPELINE PREDICTIONS
# ------------------------------------------------------------------------------
print(f"\n🔬 Running end-to-end pipeline predictions...")

pipeline_predictions = []
pipeline_true        = []
routing_log          = []
test_texts           = test_df["apso_note"].tolist()

for idx, (text, true_code) in enumerate(zip(test_texts, true_icd10_codes)):

    pred_chapter_id  = int(stage1_test_preds[idx])
    pred_chapter     = id2chapter[pred_chapter_id]
    true_chapter     = true_code[0]
    correctly_routed = (pred_chapter == true_chapter)

    if pred_chapter in SKIP_CHAPTERS:
        pred_icd10    = skip_chapter_defaults.get(pred_chapter, "UNKNOWN")
        stage2_source = "fallback"

    elif pred_chapter not in stage2_models:
        ch_train   = train_df[train_df["chapter_label"] == pred_chapter]
        pred_icd10 = ch_train["standard_icd10"].value_counts().index[0] \
                     if len(ch_train) > 0 else "UNKNOWN"
        stage2_source = "fallback_no_model"

    else:
        ch_model     = stage2_models[pred_chapter]
        ch_tokenizer = stage2_tokenizers[pred_chapter]
        ch_lmap      = stage2_label_maps[pred_chapter]
        ch_id2label  = {int(k): v for k, v in ch_lmap["id2label"].items()}

        encoding = ch_tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=cfg["max_length"],
            return_tensors="pt"
        )
        encoding = {k: v.to(device) for k, v in encoding.items()
                    if k in ["input_ids", "attention_mask", "token_type_ids"]}

        with torch.no_grad():
            outputs    = ch_model(**encoding)
            pred_id    = torch.argmax(outputs.logits, dim=-1).item()
            pred_icd10 = ch_id2label.get(pred_id, "UNKNOWN")

        stage2_source = "resolver"

    pipeline_predictions.append(pred_icd10)
    pipeline_true.append(true_code)

    routing_log.append({
        "idx":              idx,
        "true_code":        true_code,
        "true_chapter":     true_chapter,
        "pred_chapter":     pred_chapter,
        "correctly_routed": correctly_routed,
        "pred_icd10":       pred_icd10,
        "correct":          pred_icd10 == true_code,
        "stage2_source":    stage2_source,
    })

    if (idx + 1) % 100 == 0:
        correct_so_far = sum(r["correct"] for r in routing_log)
        print(f"   [{idx+1}/{len(test_texts)}] Running accuracy: "
              f"{correct_so_far/(idx+1):.1%}")

# ------------------------------------------------------------------------------
# 4. COMPUTE END-TO-END METRICS
# ------------------------------------------------------------------------------
print(f"\n📊 Computing end-to-end metrics...")

e2e_accuracy = accuracy_score(pipeline_true, pipeline_predictions)
e2e_macro_f1 = f1_score(
    pipeline_true, pipeline_predictions,
    average="macro", zero_division=0
)

n_correct         = sum(r["correct"] for r in routing_log)
n_routing_correct = sum(r["correctly_routed"] for r in routing_log)
n_routing_errors  = sum(not r["correctly_routed"] for r in routing_log)
n_stage2_correct  = sum(r["correct"] for r in routing_log
                        if r["correctly_routed"])
within_chapter_acc = n_stage2_correct / n_routing_correct \
                     if n_routing_correct > 0 else 0

print(f"\n{'='*60}")
print(f"  E-004a END-TO-END PIPELINE RESULTS")
print(f"{'='*60}")
print(f"  Test records:            {len(routing_log):,}")
print(f"{'─'*60}")
print(f"  STAGE-1 ROUTING:")
print(f"    Correctly routed:      {n_routing_correct:,} ({n_routing_correct/len(routing_log):.1%})")
print(f"    Misrouted:             {n_routing_errors:,} ({n_routing_errors/len(routing_log):.1%})")
print(f"{'─'*60}")
print(f"  STAGE-2 WITHIN-CHAPTER (on correctly routed):")
print(f"    Correct predictions:   {n_stage2_correct:,} ({within_chapter_acc:.1%})")
print(f"{'─'*60}")
print(f"  END-TO-END:")
print(f"    Accuracy:              {e2e_accuracy:.1%}")
print(f"    Macro F1:              {e2e_macro_f1:.4f}")
print(f"{'─'*60}")
print(f"  BASELINES:")
print(f"    E-002 flat:            46.9% / 0.352")
print(f"    E-003 hierarchical:    10.6% / 0.070")
print(f"{'─'*60}")
delta_e002_acc = e2e_accuracy - 0.469
delta_e002_f1  = e2e_macro_f1 - 0.352
delta_e003_acc = e2e_accuracy - 0.106
print(f"  DELTA vs E-002:          {delta_e002_acc:+.1%} / {delta_e002_f1:+.4f}")
print(f"  DELTA vs E-003:          {delta_e003_acc:+.1%}")
print(f"{'='*60}")

# ------------------------------------------------------------------------------
# 5. PER-CHAPTER BREAKDOWN
# ------------------------------------------------------------------------------
print(f"\n📊 Per-chapter end-to-end accuracy:")
print(f"   {'Ch':4s}  {'True':>6s}  {'Routed':>7s}  {'Correct':>8s}  {'E2E Acc':>8s}")
print(f"   {'─'*45}")

chapter_e2e = defaultdict(lambda: {"total": 0, "routed": 0, "correct": 0})
for r in routing_log:
    ch = r["true_chapter"]
    chapter_e2e[ch]["total"]   += 1
    chapter_e2e[ch]["routed"]  += int(r["correctly_routed"])
    chapter_e2e[ch]["correct"] += int(r["correct"])

for ch in sorted(chapter_e2e.keys()):
    stats   = chapter_e2e[ch]
    e2e_acc = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    print(f"   {ch:4s}  {stats['total']:>6,}  {stats['routed']:>7,}"
          f"  {stats['correct']:>8,}  {e2e_acc:>8.1%}")

# ------------------------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------------------------
e2e_summary = {
    "e2e_accuracy":       e2e_accuracy,
    "e2e_macro_f1":       e2e_macro_f1,
    "stage1_routing_acc": float(stage1_routing_acc),
    "within_chapter_acc": within_chapter_acc,
    "n_correct":          n_correct,
    "n_routing_correct":  n_routing_correct,
    "n_routing_errors":   n_routing_errors,
    "n_stage2_correct":   n_stage2_correct,
    "e002_accuracy":      0.469,
    "e002_macro_f1":      0.352,
    "e003_accuracy":      0.106,
    "e003_macro_f1":      0.070,
    "delta_e002_accuracy": delta_e002_acc,
    "delta_e002_macro_f1": delta_e002_f1,
    "delta_e003_accuracy": delta_e003_acc,
}

with open(EXP_DIR / "e2e_results.json", "w") as f:
    json.dump(e2e_summary, f, indent=4)

mlflow.log_metrics({
    "e2e_accuracy":       e2e_accuracy,
    "e2e_macro_f1":       e2e_macro_f1,
    "stage1_routing_acc": float(stage1_routing_acc),
    "within_chapter_acc": within_chapter_acc,
})

config.log_event(
    phase="Phase 5: End-to-End Evaluation",
    action="e2e_evaluation_complete",
    details=e2e_summary,
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 5 complete")

In [ ]:
# ==============================================================================
# PHASE 6: REGISTRY PROMOTION (E-004a)
# ==============================================================================
import json
import shutil
from datetime import datetime

print(f"🏆 Promoting E-004a artifacts to registry...")

# Save experiment config
with open(REGISTRY_DIR / "experiment_config.json", "w") as f:
    json.dump({k: v for k, v in cfg.items()
               if not isinstance(v, list)}, f, indent=4)
print(f"   ✅ Experiment config saved")

# Save final metrics
final_metrics = {
    "experiment_id":        cfg["experiment_id"],
    "experiment_name":      cfg["experiment_name"],
    "timestamp":            datetime.now().isoformat(),
    "stage1": {
        "source":           cfg["stage1_source"],
        "test_accuracy":    0.954,
        "test_macro_f1":    0.959,
    },
    "stage2": {
        "init_model":       cfg["stage2_init_model"],
        "num_resolvers":    19,
        "skip_chapters":    list(SKIP_CHAPTERS),
        "weighted_val_f1":  0.5846,
        "weighted_val_acc": 0.6911,
    },
    "end_to_end": {
        "test_accuracy":    0.6674,
        "test_macro_f1":    0.5512,
        "stage1_routing":   0.954,
        "within_chapter":   0.6985,
    },
    "baselines": {
        "e002_accuracy":    0.469,
        "e002_macro_f1":    0.352,
        "e003_accuracy":    0.106,
        "e003_macro_f1":    0.070,
    },
    "deltas": {
        "vs_e002_accuracy": 0.6674 - 0.469,
        "vs_e002_macro_f1": 0.5512 - 0.352,
        "vs_e003_accuracy": 0.6674 - 0.106,
    },
    "key_finding": (
        "E-002 initialisation for Stage-2 resolvers produced 66.7% "
        "end-to-end accuracy — +19.8pp over E-002 flat (46.9%) and "
        "+56.1pp over E-003 fresh BERT (10.6%). The hierarchical "
        "architecture with correct initialisation is definitively "
        "superior to flat ICD-10 classification."
    )
}

with open(REGISTRY_DIR / "final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=4)
print(f"   ✅ Final metrics saved")

# Copy e2e results
shutil.copy(EXP_DIR / "e2e_results.json", REGISTRY_DIR / "e2e_results.json")
print(f"   ✅ E2E results saved")

# Close MLflow run
if mlflow.active_run():
    mlflow.end_run()
    print(f"   ✅ MLflow run closed")

config.log_event(
    phase="Phase 6: Registry Promotion",
    action="e004a_registry_promotion_complete",
    details={
        "registry_path":  str(REGISTRY_DIR),
        "e2e_accuracy":   0.6674,
        "e2e_macro_f1":   0.5512,
        "mlflow_closed":  True,
    },
    notebook="05-Model_Hierarchical_ICD10_E002Init"
)

print(f"\n📝 Audit trail updated")
print(f"\n{'='*60}")
print(f"  E-004a REGISTRY SUMMARY")
print(f"{'='*60}")
print(f"  Registry:       {REGISTRY_DIR.name}/")
print(f"  Stage-1:        E-003 registry (reused)")
print(f"  Stage-2:        stage2/{{chapter}}/model/ (19 resolvers)")
print(f"  Metrics:        final_metrics.json")
print(f"{'='*60}")
print(f"  E2E Accuracy:   66.7% (+19.8pp vs E-002)")
print(f"  E2E Macro F1:   0.551 (+0.199 vs E-002)")
print(f"  Status:         COMPLETE")
print(f"{'='*60}")
print(f"\n✅ Phase 6 complete — notebook 05 done")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 E-004a Results: Interpretation

### Official E-004a Results

| Metric | Value | Context |
|---|---|---|
| End-to-end Accuracy | 66.7% | Top-1 ICD-10 classification across 1,926 codes |
| End-to-end Macro F1 | 0.551 | Equal weight to all 1,926 classes |
| Stage-1 routing accuracy | 95.4% | Identical to E-003 — same model |
| Stage-2 within-chapter accuracy | 69.8% | On correctly routed records |

---

### The Complete Picture

| Experiment | Architecture | Accuracy | Macro F1 |
|---|---|---|---|
| E-001 | ICD-3 flat, 675 classes | 82.7% | 0.760 |
| E-002 | ICD-10 flat, 1,926 classes | 46.9% | 0.352 |
| E-003 | Hierarchical, fresh BERT | 10.6% | 0.070 |
| **E-004a** | **Hierarchical, E-002 init** | **66.7%** | **0.551** |

E-004a outperforms E-002 by **+19.8pp accuracy** and **+0.199 Macro F1**.
The hierarchical architecture with the correct initialisation is
definitively superior to flat ICD-10 classification.

---

### What the E-002 Initialisation Fixed

The difference between E-003 (10.6%) and E-004a (66.7%) isolates the
effect of the Stage-2 initialisation — every other variable is identical.

**E-003 Stage-2:** Fresh Bio_ClinicalBERT trained from scratch on
chapter-filtered subsets. Had to learn ICD-10 representations from
~4 training examples per code with no prior ICD knowledge.

**E-004a Stage-2:** E-002 weights fine-tuned on chapter-filtered data.
Already maps clinical notes to 1,926 ICD-10 codes with 46.9% accuracy.
Fine-tuning sharpens within-chapter discrimination rather than learning
ICD-10 from scratch.

The within-chapter accuracy improvement tells the story clearly:
E-003 achieved 11.1% within-chapter accuracy, E-004a achieved 69.8% —
a **6.3x improvement** from the initialisation change alone.

---

### Per-Chapter Analysis

**Strongest chapters (>75% E2E accuracy):**
G (84.0%), L (79.1%), C (77.1%), B (76.9%), M (76.0%), A (75.0%)

**Good chapters (65–75%):**
R (73.5%), N (73.2%), O (72.7%), J (72.0%), K (71.4%), I (71.2%),
D (69.2%), S (69.4%), F (69.6%)

**Weaker chapters (<65%):**
- **Z (34.1%)** — 263 classes, the most ambiguous chapter in the
  dataset. Z-codes cover administrative health encounters with highly
  similar clinical language. Still 2.3x better than E-003 (14.5%).
- T (60.0%), E (62.5%), H (65.9%) — smaller chapters with more
  challenging within-class discrimination

**Skip chapters** (P: 20%, Q: 0%, U: 100%) — very small test sets,
not statistically meaningful.

---

### Convergence Analysis

The Stage-2 resolvers were not fully converged at 20 epochs — several
chapters showed val loss still decreasing. Extended training (25–30
epochs) would likely push the Z-chapter resolver and other weaker
chapters meaningfully higher.

---

### What This Means for the Pipeline

The hierarchical architecture is now validated:

1. **Stage-1** (E-001 init → 22-way chapter router) — 95.4% routing
   accuracy, substantially exceeding E-002's implicit 82.9% chapter
   accuracy
2. **Stage-2** (E-002 init → within-chapter resolvers) — 69.8%
   within-chapter accuracy, producing 66.7% end-to-end accuracy

The pipeline achieves **66.7% ICD-10 classification accuracy** on
1,926 codes from ~4 training examples per code — a strong result
for an extremely low-resource classification task.

---

### Recommendations for Next Experiments

| Experiment | Change | Expected Impact |
|---|---|---|
| E-005a | Extend Stage-2 to 30 epochs | Z-chapter and weak resolvers improve |
| E-005b | Add dialogue as supplementary input | Tests whether transcript adds signal |
| E-005c | Markdown stripping before tokenisation | Frees `**` tokens from context window |
| E-005d | Apply to real clinical dataset | Resolves MedSynth uniform sampling constraint |

> **STATUS: COMPLETE.**
> E-004a establishes the hierarchical pipeline as the official
> best-performing ICD-10 architecture.
> End-to-end Accuracy = 66.7%, Macro F1 = 0.551.
> Beats E-002 flat classifier by +19.8pp and +0.199 F1.

</div>